# SRP Checkpoint 4 - Punjenje dimenzijskog modela podataka

Punjenje prethodno izrađenog dimenzijskog modela podataka

In [21]:
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path

## Definiranje izvora podataka

2 izvora podataka:

1. relacijska MySQL baza podataka - transakcijski tj. operativni izvor podataka.

2. CSV datoteka (sadrži izvorne bankovne transakcije) - datotečni izvor podataka.

Na taj se način simulira punjenje skladišta podataka iz više različitih izvora.

In [23]:
MYSQL_USER = "root"
MYSQL_PASSWORD = "root"
MYSQL_HOST = "localhost"
MYSQL_PORT = "3306"

SOURCE_DB = "bank_fraud_db"
DW_DB = "bank_fraud_dw"

CSV_SOURCE_PATH = Path("Bank_Transaction_Fraud_Detection_PROCESSED_20.csv")

In [25]:
source_engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{SOURCE_DB}"
)

dw_engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{DW_DB}"
)

In [27]:
with source_engine.connect() as connection:
    source_tables = pd.read_sql("SHOW TABLES;", connection)

with dw_engine.connect() as connection:
    dw_tables = pd.read_sql("SHOW TABLES;", connection)

print("Tablice u relacijskoj bazi:")
display(source_tables)

print("Tablice u dimenzijskoj bazi:")
display(dw_tables)

Tablice u relacijskoj bazi:


,Tables_in_bank_fraud_db
0,account
1,account_type
2,bank_branch
3,bank_transaction
4,city
5,currency
6,customer
7,device_type
8,merchant
9,merchant_category


Tablice u dimenzijskoj bazi:


,Tables_in_bank_fraud_dw
0,dim_account
1,dim_currency
2,dim_customer
3,dim_date
4,dim_device
5,dim_merchant
6,dim_transaction_location
7,dim_transaction_type
8,fact_bank_transaction


In [29]:
csv_source_df = pd.read_csv(CSV_SOURCE_PATH)

print("Broj redaka i stupaca u CSV izvoru:")
print(csv_source_df.shape)

print("Prvih 5 redaka CSV izvora:")
display(csv_source_df.head())

Broj redaka i stupaca u CSV izvoru:
(40000, 24)
Prvih 5 redaka CSV izvora:


,Customer_ID,Customer_Name,Gender,Age,State,City,Bank_Branch,Account_Type,Transaction_ID,Transaction_Date,...,Merchant_Category,Account_Balance,Transaction_Device,Transaction_Location,Device_Type,Is_Fraud,Transaction_Currency,Customer_Contact,Transaction_Description,Customer_Email
0,88121abd-daba-484f-8618-d9b5c2afec1d,Ekavir Joshi,Female,51,Tamil Nadu,Chennai,Chennai Branch,Savings,f8265ddf-e48e-4cae-b33d-544d6460404d,2025-01-12,...,Groceries,62287.01,ATM,"Chennai, Tamil Nadu",ATM,0,INR,+9196872XXXXXX,Gifts and souvenirs,ekavirXXXX@XXXXXX.com
1,1e14f022-1a81-4000-8599-7d72c594416a,Chaman Sankar,Female,42,Nagaland,Mokokchung,Mokokchung Branch,Checking,db8902c4-57ad-4b9d-bf45-331345999cfd,2025-01-18,...,Clothing,89820.55,ATM,"Mokokchung, Nagaland",Mobile,0,INR,+9198022XXXXXX,Mobile phone payment,chamanXXX@XXXXXX.com
2,06fca2ad-25dc-40ec-8b16-d7f2ce60846b,Dayamai Swaminathan,Male,36,Jharkhand,Bokaro,Bokaro Branch,Savings,ff2a13b4-3de3-47da-939d-9725b513c8d1,2025-01-18,...,Groceries,73135.36,QR Code Scanner,"Bokaro, Jharkhand",ATM,0,INR,+9195049XXXXXX,Home repairs,dayamaiXXXX@XXXXXXX.com
3,0897e413-73b5-4c52-9e4f-63ae5f473fba,Nimrat Pal,Male,69,Goa,Vasco da Gama,Vasco da Gama Branch,Business,173d331d-e809-4d8c-8efb-b2847a912c9e,2025-01-14,...,Restaurant,93516.12,Smart Card,"Vasco da Gama, Goa",Mobile,0,INR,+9192837XXXXXX,Freelancer payment,nimratXXXX@XXXXXX.com
4,7be00d76-34e2-4ba6-aa14-d86513123c6a,Vidhi Goel,Male,59,Dadra and Nagar Haveli and Daman and Diu,Silvassa,Silvassa Branch,Savings,afe6e7d9-42da-4906-b918-cc4b6c281d60,2025-01-03,...,Electronics,99484.84,Web Browser,"Silvassa, Dadra and Nagar Haveli and Daman and...",POS,0,INR,+9198494XXXXXX,Debt repayment,vidhiXXX@XXXXXX.com


## Extract faza - izdvajanje podataka iz izvora

In [31]:
mysql_extract_query = """
SELECT
    bt.transaction_id,
    bt.transaction_code,
    bt.transaction_date,
    bt.transaction_time,
    bt.transaction_amount,
    bt.account_balance,
    bt.is_fraud,
    bt.transaction_description,

    c.customer_id,
    c.customer_code,
    c.customer_name,
    c.gender,
    c.age,
    c.customer_contact,
    c.customer_email,

    a.account_id,
    at.account_type_id,
    at.account_type_name,

    bb.bank_branch_id,
    bb.bank_branch_name,

    ci.city_id,
    ci.city_name,

    s.state_id,
    s.state_name,

    m.merchant_id,
    m.merchant_code,

    mc.merchant_category_id,
    mc.merchant_category_name,

    tt.transaction_type_id,
    tt.transaction_type_name,

    td.transaction_device_id,
    td.transaction_device_name,

    dt.device_type_id,
    dt.device_type_name,

    tl.transaction_location_id,
    tl.transaction_location_name,

    cur.currency_id,
    cur.currency_code

FROM bank_transaction bt
LEFT JOIN account a 
    ON bt.account_id = a.account_id
LEFT JOIN customer c 
    ON a.customer_id = c.customer_id
LEFT JOIN account_type at 
    ON a.account_type_id = at.account_type_id
LEFT JOIN bank_branch bb 
    ON a.bank_branch_id = bb.bank_branch_id
LEFT JOIN city ci 
    ON bb.city_id = ci.city_id
LEFT JOIN state s 
    ON ci.state_id = s.state_id
LEFT JOIN merchant m 
    ON bt.merchant_id = m.merchant_id
LEFT JOIN merchant_category mc 
    ON m.merchant_category_id = mc.merchant_category_id
LEFT JOIN transaction_type tt 
    ON bt.transaction_type_id = tt.transaction_type_id
LEFT JOIN transaction_device td 
    ON bt.transaction_device_id = td.transaction_device_id
LEFT JOIN device_type dt 
    ON td.device_type_id = dt.device_type_id
LEFT JOIN transaction_location tl 
    ON bt.transaction_location_id = tl.transaction_location_id
LEFT JOIN currency cur 
    ON bt.currency_id = cur.currency_id;
"""

In [33]:
with source_engine.connect() as connection:
    mysql_source_df = pd.read_sql(text(mysql_extract_query), connection)

print("Broj redaka i stupaca dohvaćenih iz MySQL izvora:")
print(mysql_source_df.shape)

display(mysql_source_df.head())

Broj redaka i stupaca dohvaćenih iz MySQL izvora:
(160000, 38)


,transaction_id,transaction_code,transaction_date,transaction_time,transaction_amount,account_balance,is_fraud,transaction_description,customer_id,customer_code,...,transaction_type_id,transaction_type_name,transaction_device_id,transaction_device_name,device_type_id,device_type_name,transaction_location_id,transaction_location_name,currency_id,currency_code
0,1,4fa3208f-9e23-42dc-b330-844829d0c12c,2025-01-23,0 days 16:04:07,32415.45,74557.27,0,Bitcoin transaction,1,d5f6ec07-d69e-4f47-b9b4-7c58ff17c19e,...,1,Transfer,1,Voice Assistant,1,POS,1,"Thiruvananthapuram, Kerala",1,INR
1,2,e41c55f9-c016-4ff3-872b-cae72467c75c,2025-01-25,0 days 03:09:52,63062.56,66817.99,0,Mutual fund investment,2,3a73a0e5-d4da-45aa-85f3-528413900a35,...,2,Bill Payment,2,ATM,2,Desktop,2,"Bhagalpur, Bihar",1,INR
2,3,af5f667c-d064-4083-bfb7-83396111a3da,2025-01-25,0 days 06:49:53,9711.15,61258.85,0,Seminar registration,3,6c870d65-76b0-431d-bdf3-9292998e8211,...,1,Transfer,3,Mobile Device,1,POS,3,"Ahmedabad, Gujarat",1,INR
3,4,b1355810-d246-4aeb-9932-347f32646172,2025-01-04,0 days 00:53:33,94677.01,36313.61,0,Public transport pass,4,5323737c-bbd2-423f-9c9b-e0433c8f75dc,...,1,Transfer,4,Payment Gateway Device,2,Desktop,4,"New Delhi, Delhi",1,INR
4,5,c86a000c-d81f-40be-acdf-77fc072fd808,2025-01-16,0 days 04:04:48,67704.28,16948.73,0,Online shopping,5,c0c3d474-f6c2-4c66-9b0e-f9ba75c6f310,...,3,Debit,5,Debit/Credit Card,3,ATM,5,"Port Blair, Andaman and Nicobar Islands",1,INR


### Izdvajanje podataka iz CSV izvora

In [36]:
print("Broj redaka i stupaca dohvaćenih iz CSV izvora:")
print(csv_source_df.shape)

display(csv_source_df.head())

Broj redaka i stupaca dohvaćenih iz CSV izvora:
(40000, 24)


,Customer_ID,Customer_Name,Gender,Age,State,City,Bank_Branch,Account_Type,Transaction_ID,Transaction_Date,...,Merchant_Category,Account_Balance,Transaction_Device,Transaction_Location,Device_Type,Is_Fraud,Transaction_Currency,Customer_Contact,Transaction_Description,Customer_Email
0,88121abd-daba-484f-8618-d9b5c2afec1d,Ekavir Joshi,Female,51,Tamil Nadu,Chennai,Chennai Branch,Savings,f8265ddf-e48e-4cae-b33d-544d6460404d,2025-01-12,...,Groceries,62287.01,ATM,"Chennai, Tamil Nadu",ATM,0,INR,+9196872XXXXXX,Gifts and souvenirs,ekavirXXXX@XXXXXX.com
1,1e14f022-1a81-4000-8599-7d72c594416a,Chaman Sankar,Female,42,Nagaland,Mokokchung,Mokokchung Branch,Checking,db8902c4-57ad-4b9d-bf45-331345999cfd,2025-01-18,...,Clothing,89820.55,ATM,"Mokokchung, Nagaland",Mobile,0,INR,+9198022XXXXXX,Mobile phone payment,chamanXXX@XXXXXX.com
2,06fca2ad-25dc-40ec-8b16-d7f2ce60846b,Dayamai Swaminathan,Male,36,Jharkhand,Bokaro,Bokaro Branch,Savings,ff2a13b4-3de3-47da-939d-9725b513c8d1,2025-01-18,...,Groceries,73135.36,QR Code Scanner,"Bokaro, Jharkhand",ATM,0,INR,+9195049XXXXXX,Home repairs,dayamaiXXXX@XXXXXXX.com
3,0897e413-73b5-4c52-9e4f-63ae5f473fba,Nimrat Pal,Male,69,Goa,Vasco da Gama,Vasco da Gama Branch,Business,173d331d-e809-4d8c-8efb-b2847a912c9e,2025-01-14,...,Restaurant,93516.12,Smart Card,"Vasco da Gama, Goa",Mobile,0,INR,+9192837XXXXXX,Freelancer payment,nimratXXXX@XXXXXX.com
4,7be00d76-34e2-4ba6-aa14-d86513123c6a,Vidhi Goel,Male,59,Dadra and Nagar Haveli and Daman and Diu,Silvassa,Silvassa Branch,Savings,afe6e7d9-42da-4906-b918-cc4b6c281d60,2025-01-03,...,Electronics,99484.84,Web Browser,"Silvassa, Dadra and Nagar Haveli and Daman and...",POS,0,INR,+9198494XXXXXX,Debt repayment,vidhiXXX@XXXXXX.com


In [38]:
print("Nazivi stupaca u CSV izvoru:")
print(csv_source_df.columns.tolist())

Nazivi stupaca u CSV izvoru:
['Customer_ID', 'Customer_Name', 'Gender', 'Age', 'State', 'City', 'Bank_Branch', 'Account_Type', 'Transaction_ID', 'Transaction_Date', 'Transaction_Time', 'Transaction_Amount', 'Merchant_ID', 'Transaction_Type', 'Merchant_Category', 'Account_Balance', 'Transaction_Device', 'Transaction_Location', 'Device_Type', 'Is_Fraud', 'Transaction_Currency', 'Customer_Contact', 'Transaction_Description', 'Customer_Email']


### Odabir potrebnih stupaca iz CSV izvora

Stupac `Transaction_ID` preimenuje se u `transaction_code` kako bi imao isti naziv kao odgovarajući stupac iz MySQL izvora.

In [41]:
csv_20_df = csv_source_df.copy()

csv_20_df["Transaction_ID"] = csv_20_df["Transaction_ID"].astype(str)
mysql_source_df["transaction_code"] = mysql_source_df["transaction_code"].astype(str)

def build_id_mapping(existing_df, existing_key_columns, existing_id_column, new_df, new_key_columns):
    existing_keys = existing_df[existing_key_columns + [existing_id_column]].drop_duplicates(subset=existing_key_columns).copy()
    existing_keys["_key"] = list(map(tuple, existing_keys[existing_key_columns].astype(str).values))
    existing_map = existing_keys.set_index("_key")[existing_id_column].to_dict()

    next_id = int(existing_df[existing_id_column].max()) + 1

    new_keys = new_df[new_key_columns].drop_duplicates().copy()
    new_keys["_key"] = list(map(tuple, new_keys[new_key_columns].astype(str).values))

    missing_keys = [key for key in new_keys["_key"].tolist() if key not in existing_map]
    generated_map = {key: next_id + index for index, key in enumerate(missing_keys)}

    return {**existing_map, **generated_map}

def map_ids(new_df, key_columns, id_map):
    keys = list(map(tuple, new_df[key_columns].astype(str).values))
    return pd.Series(keys).map(id_map).astype(int).values

transaction_id_map = build_id_mapping(mysql_source_df, ["transaction_code"], "transaction_id", csv_20_df, ["Transaction_ID"])
customer_id_map = build_id_mapping(mysql_source_df, ["customer_code"], "customer_id", csv_20_df, ["Customer_ID"])
account_id_map = build_id_mapping(mysql_source_df, ["customer_code", "account_type_name", "bank_branch_name", "city_name", "state_name"], "account_id", csv_20_df, ["Customer_ID", "Account_Type", "Bank_Branch", "City", "State"])
account_type_id_map = build_id_mapping(mysql_source_df, ["account_type_name"], "account_type_id", csv_20_df, ["Account_Type"])
bank_branch_id_map = build_id_mapping(mysql_source_df, ["bank_branch_name", "city_name", "state_name"], "bank_branch_id", csv_20_df, ["Bank_Branch", "City", "State"])
city_id_map = build_id_mapping(mysql_source_df, ["city_name", "state_name"], "city_id", csv_20_df, ["City", "State"])
state_id_map = build_id_mapping(mysql_source_df, ["state_name"], "state_id", csv_20_df, ["State"])
merchant_id_map = build_id_mapping(mysql_source_df, ["merchant_code"], "merchant_id", csv_20_df, ["Merchant_ID"])
merchant_category_id_map = build_id_mapping(mysql_source_df, ["merchant_category_name"], "merchant_category_id", csv_20_df, ["Merchant_Category"])
transaction_type_id_map = build_id_mapping(mysql_source_df, ["transaction_type_name"], "transaction_type_id", csv_20_df, ["Transaction_Type"])
transaction_device_id_map = build_id_mapping(mysql_source_df, ["transaction_device_name"], "transaction_device_id", csv_20_df, ["Transaction_Device"])
device_type_id_map = build_id_mapping(mysql_source_df, ["device_type_name"], "device_type_id", csv_20_df, ["Device_Type"])
transaction_location_id_map = build_id_mapping(mysql_source_df, ["transaction_location_name"], "transaction_location_id", csv_20_df, ["Transaction_Location"])
currency_id_map = build_id_mapping(mysql_source_df, ["currency_code"], "currency_id", csv_20_df, ["Transaction_Currency"])

csv_source_aligned_df = pd.DataFrame({
    "transaction_id": map_ids(csv_20_df, ["Transaction_ID"], transaction_id_map),
    "transaction_code": csv_20_df["Transaction_ID"].astype(str),
    "transaction_date": pd.to_datetime(csv_20_df["Transaction_Date"], format="%d-%m-%Y", errors="coerce"),
    "transaction_time": csv_20_df["Transaction_Time"].astype(str),
    "transaction_amount": csv_20_df["Transaction_Amount"].astype(float),
    "account_balance": csv_20_df["Account_Balance"].astype(float),
    "is_fraud": csv_20_df["Is_Fraud"].astype(int),
    "transaction_description": csv_20_df["Transaction_Description"],

    "customer_id": map_ids(csv_20_df, ["Customer_ID"], customer_id_map),
    "customer_code": csv_20_df["Customer_ID"].astype(str),
    "customer_name": csv_20_df["Customer_Name"],
    "gender": csv_20_df["Gender"],
    "age": csv_20_df["Age"].astype(int),
    "customer_contact": csv_20_df["Customer_Contact"],
    "customer_email": csv_20_df["Customer_Email"],

    "account_id": map_ids(csv_20_df, ["Customer_ID", "Account_Type", "Bank_Branch", "City", "State"], account_id_map),
    "account_type_id": map_ids(csv_20_df, ["Account_Type"], account_type_id_map),
    "account_type_name": csv_20_df["Account_Type"],

    "bank_branch_id": map_ids(csv_20_df, ["Bank_Branch", "City", "State"], bank_branch_id_map),
    "bank_branch_name": csv_20_df["Bank_Branch"],

    "city_id": map_ids(csv_20_df, ["City", "State"], city_id_map),
    "city_name": csv_20_df["City"],

    "state_id": map_ids(csv_20_df, ["State"], state_id_map),
    "state_name": csv_20_df["State"],

    "merchant_id": map_ids(csv_20_df, ["Merchant_ID"], merchant_id_map),
    "merchant_code": csv_20_df["Merchant_ID"].astype(str),

    "merchant_category_id": map_ids(csv_20_df, ["Merchant_Category"], merchant_category_id_map),
    "merchant_category_name": csv_20_df["Merchant_Category"],

    "transaction_type_id": map_ids(csv_20_df, ["Transaction_Type"], transaction_type_id_map),
    "transaction_type_name": csv_20_df["Transaction_Type"],

    "transaction_device_id": map_ids(csv_20_df, ["Transaction_Device"], transaction_device_id_map),
    "transaction_device_name": csv_20_df["Transaction_Device"],

    "device_type_id": map_ids(csv_20_df, ["Device_Type"], device_type_id_map),
    "device_type_name": csv_20_df["Device_Type"],

    "transaction_location_id": map_ids(csv_20_df, ["Transaction_Location"], transaction_location_id_map),
    "transaction_location_name": csv_20_df["Transaction_Location"],

    "currency_id": map_ids(csv_20_df, ["Transaction_Currency"], currency_id_map),
    "currency_code": csv_20_df["Transaction_Currency"]
})

print("Broj redaka i stupaca nakon pripreme CSV 20% izvora:")
print(csv_source_aligned_df.shape)

display(csv_source_aligned_df.head())

Broj redaka i stupaca nakon pripreme CSV 20% izvora:
(40000, 38)


,transaction_id,transaction_code,transaction_date,transaction_time,transaction_amount,account_balance,is_fraud,transaction_description,customer_id,customer_code,...,transaction_type_id,transaction_type_name,transaction_device_id,transaction_device_name,device_type_id,device_type_name,transaction_location_id,transaction_location_name,currency_id,currency_code
0,160001,f8265ddf-e48e-4cae-b33d-544d6460404d,NaT,08:46:40,96571.40,62287.01,0,Gifts and souvenirs,160001,88121abd-daba-484f-8618-d9b5c2afec1d,...,4,Withdrawal,2,ATM,3,ATM,96,"Chennai, Tamil Nadu",1,INR
1,160002,db8902c4-57ad-4b9d-bf45-331345999cfd,NaT,17:53:01,22657.92,89820.55,0,Mobile phone payment,160002,1e14f022-1a81-4000-8599-7d72c594416a,...,4,Withdrawal,2,ATM,4,Mobile,68,"Mokokchung, Nagaland",1,INR
2,160003,ff2a13b4-3de3-47da-939d-9725b513c8d1,NaT,09:27:50,4075.19,73135.36,0,Home repairs,160003,06fca2ad-25dc-40ec-8b16-d7f2ce60846b,...,2,Bill Payment,14,QR Code Scanner,3,ATM,36,"Bokaro, Jharkhand",1,INR
3,160004,173d331d-e809-4d8c-8efb-b2847a912c9e,NaT,20:56:30,31608.04,93516.12,0,Freelancer payment,160004,0897e413-73b5-4c52-9e4f-63ae5f473fba,...,1,Transfer,15,Smart Card,4,Mobile,59,"Vasco da Gama, Goa",1,INR
4,160005,afe6e7d9-42da-4906-b918-cc4b6c281d60,NaT,10:47:57,79892.86,99484.84,0,Debt repayment,160005,7be00d76-34e2-4ba6-aa14-d86513123c6a,...,5,Credit,12,Web Browser,1,POS,13,"Silvassa, Dadra and Nagar Haveli and Daman and...",1,INR


In [43]:
mysql_duplicates = mysql_source_df["transaction_code"].duplicated().sum()
csv_duplicates = csv_source_aligned_df["transaction_code"].duplicated().sum()

mysql_codes = set(mysql_source_df["transaction_code"].astype(str))
csv_codes = set(csv_source_aligned_df["transaction_code"].astype(str))

overlap_count = len(mysql_codes.intersection(csv_codes))

print("Broj duplikata u MySQL 80% izvoru po transaction_code:", mysql_duplicates)
print("Broj duplikata u CSV 20% izvoru po transaction_code:", csv_duplicates)
print("Broj transakcija koje se pojavljuju u oba izvora:", overlap_count)

if mysql_duplicates == 0 and csv_duplicates == 0 and overlap_count == 0:
    print("MySQL 80% i CSV 20% su odvojeni izvori i mogu se spojiti dodavanjem redaka.")
else:
    print("Potrebno je provjeriti duplikate ili preklapanje između izvora.")

Broj duplikata u MySQL 80% izvoru po transaction_code: 0
Broj duplikata u CSV 20% izvoru po transaction_code: 0
Broj transakcija koje se pojavljuju u oba izvora: 0
MySQL 80% i CSV 20% su odvojeni izvori i mogu se spojiti dodavanjem redaka.


Ne postoje duplikati po stupcu `transaction_code` -> može se sigurno koristiti kao ključ za povezivanje dva izvora

### Spajanje izvora u staging dataframe

In [57]:
mysql_staging_part = mysql_source_df.copy()
mysql_staging_part["source_system"] = "MySQL 80%"

csv_staging_part = csv_source_aligned_df.copy()
csv_staging_part["source_system"] = "CSV 20%"

staging_df = pd.concat(
    [mysql_staging_part, csv_staging_part],
    ignore_index=True
)

print("Broj redaka i stupaca nakon spajanja dva izvora u staging dataframe:")
print(staging_df.shape)

display(staging_df.head())

Broj redaka i stupaca nakon spajanja dva izvora u staging dataframe:
(200000, 39)


,transaction_id,transaction_code,transaction_date,transaction_time,transaction_amount,account_balance,is_fraud,transaction_description,customer_id,customer_code,...,transaction_type_name,transaction_device_id,transaction_device_name,device_type_id,device_type_name,transaction_location_id,transaction_location_name,currency_id,currency_code,source_system
0,1,4fa3208f-9e23-42dc-b330-844829d0c12c,2025-01-23,0 days 16:04:07,32415.45,74557.27,0,Bitcoin transaction,1,d5f6ec07-d69e-4f47-b9b4-7c58ff17c19e,...,Transfer,1,Voice Assistant,1,POS,1,"Thiruvananthapuram, Kerala",1,INR,MySQL 80%
1,2,e41c55f9-c016-4ff3-872b-cae72467c75c,2025-01-25,0 days 03:09:52,63062.56,66817.99,0,Mutual fund investment,2,3a73a0e5-d4da-45aa-85f3-528413900a35,...,Bill Payment,2,ATM,2,Desktop,2,"Bhagalpur, Bihar",1,INR,MySQL 80%
2,3,af5f667c-d064-4083-bfb7-83396111a3da,2025-01-25,0 days 06:49:53,9711.15,61258.85,0,Seminar registration,3,6c870d65-76b0-431d-bdf3-9292998e8211,...,Transfer,3,Mobile Device,1,POS,3,"Ahmedabad, Gujarat",1,INR,MySQL 80%
3,4,b1355810-d246-4aeb-9932-347f32646172,2025-01-04,0 days 00:53:33,94677.01,36313.61,0,Public transport pass,4,5323737c-bbd2-423f-9c9b-e0433c8f75dc,...,Transfer,4,Payment Gateway Device,2,Desktop,4,"New Delhi, Delhi",1,INR,MySQL 80%
4,5,c86a000c-d81f-40be-acdf-77fc072fd808,2025-01-16,0 days 04:04:48,67704.28,16948.73,0,Online shopping,5,c0c3d474-f6c2-4c66-9b0e-f9ba75c6f310,...,Debit,5,Debit/Credit Card,3,ATM,5,"Port Blair, Andaman and Nicobar Islands",1,INR,MySQL 80%


In [59]:
mysql_row_count = len(mysql_source_df)
csv_row_count = len(csv_source_aligned_df)
staging_row_count = len(staging_df)
expected_row_count = mysql_row_count + csv_row_count

print("Broj redaka u MySQL 80% izvoru:", mysql_row_count)
print("Broj redaka u CSV 20% izvoru:", csv_row_count)
print("Očekivani broj redaka nakon spajanja:", expected_row_count)
print("Broj redaka u staging dataframeu:", staging_row_count)

display(staging_df["source_system"].value_counts())

if staging_row_count == expected_row_count:
    print("Svi redci iz oba izvora uspješno su uključeni u staging dataframe.")
else:
    print("Postoji razlika u broju redaka. Potrebno je provjeriti spajanje izvora.")

Broj redaka u MySQL 80% izvoru: 160000
Broj redaka u CSV 20% izvoru: 40000
Očekivani broj redaka nakon spajanja: 200000
Broj redaka u staging dataframeu: 200000


source_system
MySQL 80%    160000
CSV 20%       40000
Name: count, dtype: int64

Svi redci iz oba izvora uspješno su uključeni u staging dataframe.


### Provjera konzistentnosti podataka između izvora

In [61]:
required_columns = [
    "transaction_id",
    "transaction_code",
    "transaction_date",
    "transaction_time",
    "transaction_amount",
    "account_balance",
    "is_fraud",
    "customer_id",
    "account_id",
    "merchant_id",
    "transaction_type_id",
    "transaction_device_id",
    "transaction_location_id",
    "currency_id"
]

missing_required_values = staging_df[required_columns].isna().sum()
missing_required_values = missing_required_values[missing_required_values > 0]

print("Nedostajuće vrijednosti u ključnim stupcima staging dataframea:")
display(missing_required_values)

if missing_required_values.empty:
    print("Svi ključni stupci su popunjeni za oba izvora.")
else:
    print("Postoje nedostajuće vrijednosti koje treba provjeriti prije Transform faze.")

Nedostajuće vrijednosti u ključnim stupcima staging dataframea:


Series([], dtype: int64)

Svi ključni stupci su popunjeni za oba izvora.


In [53]:
# prvotno je bilo nedostajucih u transaction date

print("Primjeri vrijednosti iz stupca Transaction_Date u CSV 20% izvoru:")
display(csv_20_df["Transaction_Date"].head(10))

print("Tip stupca Transaction_Date:")
print(csv_20_df["Transaction_Date"].dtype)

print("Broj nedostajućih vrijednosti u originalnom CSV stupcu Transaction_Date:")
print(csv_20_df["Transaction_Date"].isna().sum())

Primjeri vrijednosti iz stupca Transaction_Date u CSV 20% izvoru:


0    2025-01-12
1    2025-01-18
2    2025-01-18
3    2025-01-14
4    2025-01-03
5    2025-01-03
6    2025-01-05
7    2025-01-08
8    2025-01-27
9    2025-01-16
Name: Transaction_Date, dtype: object

Tip stupca Transaction_Date:
object
Broj nedostajućih vrijednosti u originalnom CSV stupcu Transaction_Date:
0


In [55]:
# razlog je krivi format, sad popravljanje

csv_source_aligned_df["transaction_date"] = pd.to_datetime(
    csv_20_df["Transaction_Date"],
    format="%Y-%m-%d",
    errors="coerce"
)

print("Broj neprepoznatih datuma nakon popravka:")
print(csv_source_aligned_df["transaction_date"].isna().sum())

print("Primjeri popravljenih datuma:")
display(csv_source_aligned_df[["transaction_code", "transaction_date"]].head(10))

# ponovno na staging dio: mysql_staging_part = mysql_source_df.copy()

Broj neprepoznatih datuma nakon popravka:
0
Primjeri popravljenih datuma:


,transaction_code,transaction_date
0,f8265ddf-e48e-4cae-b33d-544d6460404d,2025-01-12
1,db8902c4-57ad-4b9d-bf45-331345999cfd,2025-01-18
2,ff2a13b4-3de3-47da-939d-9725b513c8d1,2025-01-18
3,173d331d-e809-4d8c-8efb-b2847a912c9e,2025-01-14
4,afe6e7d9-42da-4906-b918-cc4b6c281d60,2025-01-03
5,2f93e56c-0e14-4016-a4b7-0a8e67983180,2025-01-03
6,613826f1-76e8-4d0f-8c37-57bdc62adf53,2025-01-05
7,23f1e3ef-abbf-463e-9ede-92713c2e44a0,2025-01-08
8,a39710c3-3b56-485a-b67d-dd47f22c1021,2025-01-27
9,1bb528a8-8c45-4c05-9879-580212143eb1,2025-01-16


CSV izvor uspješno je povezan s MySQL izvorom

## Transform faza - čišćenje i priprema podataka

`staging_df` kao pripremno područje podataka.

In [64]:
etl_df = staging_df.copy()

print("Broj redaka i stupaca prije transformacije:")
print(etl_df.shape)

display(etl_df.head())

Broj redaka i stupaca prije transformacije:
(200000, 39)


,transaction_id,transaction_code,transaction_date,transaction_time,transaction_amount,account_balance,is_fraud,transaction_description,customer_id,customer_code,...,transaction_type_name,transaction_device_id,transaction_device_name,device_type_id,device_type_name,transaction_location_id,transaction_location_name,currency_id,currency_code,source_system
0,1,4fa3208f-9e23-42dc-b330-844829d0c12c,2025-01-23,0 days 16:04:07,32415.45,74557.27,0,Bitcoin transaction,1,d5f6ec07-d69e-4f47-b9b4-7c58ff17c19e,...,Transfer,1,Voice Assistant,1,POS,1,"Thiruvananthapuram, Kerala",1,INR,MySQL 80%
1,2,e41c55f9-c016-4ff3-872b-cae72467c75c,2025-01-25,0 days 03:09:52,63062.56,66817.99,0,Mutual fund investment,2,3a73a0e5-d4da-45aa-85f3-528413900a35,...,Bill Payment,2,ATM,2,Desktop,2,"Bhagalpur, Bihar",1,INR,MySQL 80%
2,3,af5f667c-d064-4083-bfb7-83396111a3da,2025-01-25,0 days 06:49:53,9711.15,61258.85,0,Seminar registration,3,6c870d65-76b0-431d-bdf3-9292998e8211,...,Transfer,3,Mobile Device,1,POS,3,"Ahmedabad, Gujarat",1,INR,MySQL 80%
3,4,b1355810-d246-4aeb-9932-347f32646172,2025-01-04,0 days 00:53:33,94677.01,36313.61,0,Public transport pass,4,5323737c-bbd2-423f-9c9b-e0433c8f75dc,...,Transfer,4,Payment Gateway Device,2,Desktop,4,"New Delhi, Delhi",1,INR,MySQL 80%
4,5,c86a000c-d81f-40be-acdf-77fc072fd808,2025-01-16,0 days 04:04:48,67704.28,16948.73,0,Online shopping,5,c0c3d474-f6c2-4c66-9b0e-f9ba75c6f310,...,Debit,5,Debit/Credit Card,3,ATM,5,"Port Blair, Andaman and Nicobar Islands",1,INR,MySQL 80%


### Pretvorba datuma i vremena

In [66]:
etl_df["transaction_date"] = pd.to_datetime(etl_df["transaction_date"])

def format_transaction_time(value):
    if pd.isna(value):
        return None
    
    if isinstance(value, pd.Timedelta):
        total_seconds = int(value.total_seconds())
        hours = total_seconds // 3600
        minutes = (total_seconds % 3600) // 60
        seconds = total_seconds % 60
        return f"{hours:02d}:{minutes:02d}:{seconds:02d}"
    
    value_str = str(value)
    
    if "days" in value_str:
        value_str = value_str.split("days")[-1].strip()
    
    return value_str

etl_df["transaction_time"] = etl_df["transaction_time"].apply(format_transaction_time)

display(etl_df[["transaction_date", "transaction_time"]].head())

,transaction_date,transaction_time
0,2025-01-23,16:04:07
1,2025-01-25,03:09:52
2,2025-01-25,06:49:53
3,2025-01-04,00:53:33
4,2025-01-16,04:04:48


### Uklanjanje pomoćnih stupaca iz CSV izvora

Nema razlika između vrijednosti iz CSV i MySQL izvora -> pomoćni CSV stupci više nisu potrebni za daljnju transformaciju

In [69]:
etl_df = etl_df.drop(columns=["source_system"], errors="ignore")

print("Broj redaka i stupaca nakon uklanjanja pomoćnog stupca source_system:")
print(etl_df.shape)

display(etl_df.head())

Broj redaka i stupaca nakon uklanjanja pomoćnog stupca source_system:
(200000, 38)


,transaction_id,transaction_code,transaction_date,transaction_time,transaction_amount,account_balance,is_fraud,transaction_description,customer_id,customer_code,...,transaction_type_id,transaction_type_name,transaction_device_id,transaction_device_name,device_type_id,device_type_name,transaction_location_id,transaction_location_name,currency_id,currency_code
0,1,4fa3208f-9e23-42dc-b330-844829d0c12c,2025-01-23,16:04:07,32415.45,74557.27,0,Bitcoin transaction,1,d5f6ec07-d69e-4f47-b9b4-7c58ff17c19e,...,1,Transfer,1,Voice Assistant,1,POS,1,"Thiruvananthapuram, Kerala",1,INR
1,2,e41c55f9-c016-4ff3-872b-cae72467c75c,2025-01-25,03:09:52,63062.56,66817.99,0,Mutual fund investment,2,3a73a0e5-d4da-45aa-85f3-528413900a35,...,2,Bill Payment,2,ATM,2,Desktop,2,"Bhagalpur, Bihar",1,INR
2,3,af5f667c-d064-4083-bfb7-83396111a3da,2025-01-25,06:49:53,9711.15,61258.85,0,Seminar registration,3,6c870d65-76b0-431d-bdf3-9292998e8211,...,1,Transfer,3,Mobile Device,1,POS,3,"Ahmedabad, Gujarat",1,INR
3,4,b1355810-d246-4aeb-9932-347f32646172,2025-01-04,00:53:33,94677.01,36313.61,0,Public transport pass,4,5323737c-bbd2-423f-9c9b-e0433c8f75dc,...,1,Transfer,4,Payment Gateway Device,2,Desktop,4,"New Delhi, Delhi",1,INR
4,5,c86a000c-d81f-40be-acdf-77fc072fd808,2025-01-16,04:04:48,67704.28,16948.73,0,Online shopping,5,c0c3d474-f6c2-4c66-9b0e-f9ba75c6f310,...,3,Debit,5,Debit/Credit Card,3,ATM,5,"Port Blair, Andaman and Nicobar Islands",1,INR


### Provjera nedostajućih vrijednosti

In [71]:
missing_values = etl_df.isna().sum()
missing_values = missing_values[missing_values > 0]

print("Stupci s nedostajućim vrijednostima:")
display(missing_values)

Stupci s nedostajućim vrijednostima:


Series([], dtype: int64)

### Provjera tipova podataka

In [73]:
print("Tipovi podataka nakon osnovne transformacije:")
display(etl_df.dtypes)

Tipovi podataka nakon osnovne transformacije:


transaction_id                        int64
transaction_code                     object
transaction_date             datetime64[ns]
transaction_time                     object
transaction_amount                  float64
account_balance                     float64
is_fraud                              int64
transaction_description              object
customer_id                           int64
customer_code                        object
customer_name                        object
gender                               object
age                                   int64
customer_contact                     object
customer_email                       object
account_id                            int64
account_type_id                       int64
account_type_name                    object
bank_branch_id                        int64
bank_branch_name                     object
city_id                               int64
city_name                            object
state_id                        

### Provjera numeričkih vrijednosti

In [75]:
numeric_summary = etl_df[
    [
        "transaction_amount",
        "account_balance",
        "is_fraud"
    ]
].describe()

display(numeric_summary)

,transaction_amount,account_balance,is_fraud
count,200000.000000,200000.000000,200000.000000
mean,49538.015554,52437.988784,0.050440
std,28551.874004,27399.507128,0.218852
min,10.290000,5000.820000,0.000000
25%,24851.345000,28742.395000,0.000000
50%,49502.440000,52372.555000,0.000000
75%,74314.625000,76147.670000,0.000000
max,98999.980000,99999.950000,1.000000


### Provjera broja redaka nakon Transform faze

je li broj redaka ostao isti kao u staging dataframeu?

In [77]:
print("Broj redaka u staging dataframeu:", len(staging_df))
print("Broj redaka nakon transformacije:", len(etl_df))

if len(staging_df) == len(etl_df):
    print("Nijedan redak nije izgubljen tijekom Transform faze.")
else:
    print("Postoji razlika u broju redaka nakon Transform faze.")

Broj redaka u staging dataframeu: 200000
Broj redaka nakon transformacije: 200000
Nijedan redak nije izgubljen tijekom Transform faze.


## Provjera strukture dimenzijskog modela

In [79]:
dw_table_names = [
    "dim_date",
    "dim_customer",
    "dim_account",
    "dim_merchant",
    "dim_device",
    "dim_transaction_type",
    "dim_transaction_location",
    "dim_currency",
    "fact_bank_transaction"
]

with dw_engine.connect() as connection:
    for table_name in dw_table_names:
        print(f"\nStruktura tablice: {table_name}")
        table_structure = pd.read_sql(f"SHOW COLUMNS FROM {table_name};", connection)
        display(table_structure)


Struktura tablice: dim_date


,Field,Type,Null,Key,Default,Extra
0,date_tk,int,NO,PRI,None,auto_increment
1,full_date,date,NO,UNI,None,
2,day_of_month,int,YES,,None,
3,month_number,int,YES,,None,
4,month_name,varchar(20),YES,,None,
5,quarter_number,int,YES,,None,
6,year_number,int,YES,,None,



Struktura tablice: dim_customer


,Field,Type,Null,Key,Default,Extra
0,customer_tk,bigint,NO,PRI,None,auto_increment
1,customer_id,int,NO,,None,
2,customer_code,varchar(100),YES,,None,
3,customer_name,varchar(100),YES,,None,
4,gender,varchar(10),YES,,None,
5,age,int,YES,,None,
6,age_group,varchar(20),YES,,None,
7,customer_contact,varchar(50),YES,,None,
8,customer_email,varchar(100),YES,,None,
9,version,int,YES,,1,



Struktura tablice: dim_account


...

,Field,Type,Null,Key,Default,Extra
0,transaction_location_tk,bigint,NO,PRI,None,auto_increment
1,transaction_location_id,int,NO,,None,
2,transaction_location_name,varchar(150),YES,,None,



Struktura tablice: dim_currency


,Field,Type,Null,Key,Default,Extra
0,currency_tk,bigint,NO,PRI,None,auto_increment
1,currency_id,int,NO,,None,
2,currency_code,varchar(10),YES,,None,



Struktura tablice: fact_bank_transaction


,Field,Type,Null,Key,Default,Extra
0,fact_transaction_tk,bigint,NO,PRI,None,auto_increment
1,transaction_id,int,NO,,None,
2,transaction_code,varchar(100),YES,,None,
3,date_tk,int,YES,MUL,None,
4,customer_tk,bigint,YES,MUL,None,
5,account_tk,bigint,YES,MUL,None,
6,merchant_tk,bigint,YES,MUL,None,
7,device_tk,bigint,YES,MUL,None,
8,transaction_type_tk,bigint,YES,MUL,None,
9,transaction_location_tk,bigint,YES,MUL,None,


## Izrada dimenzijskih dataframeova

Izrađuju se dataframeovi koji odgovaraju dimenzijskim tablicama u bazi `bank_fraud_dw`

Surogat ključevi tipa `customer_tk`, `account_tk`, `date_tk` i ostalih ne dodaju se ručno jer su u bazi definirani kao `AUTO_INCREMENT`!!

### Priprema pomoćnih atributa

Dodaje se atribut `age_group` u dim kupca, koji grupira kupce prema dobi.

In [83]:
def assign_age_group(age):
    if age < 18:
        return "Under 18"
    elif age <= 25:
        return "18-25"
    elif age <= 35:
        return "26-35"
    elif age <= 45:
        return "36-45"
    elif age <= 60:
        return "46-60"
    else:
        return "60+"

date_from_value = etl_df["transaction_date"].min().date()

### Dimenzija datuma

In [86]:
dim_date_df = etl_df[["transaction_date"]].drop_duplicates().copy()

dim_date_df = dim_date_df.rename(columns={
    "transaction_date": "full_date"
})

dim_date_df["day_of_month"] = dim_date_df["full_date"].dt.day
dim_date_df["month_number"] = dim_date_df["full_date"].dt.month
dim_date_df["month_name"] = dim_date_df["full_date"].dt.month_name()
dim_date_df["quarter_number"] = dim_date_df["full_date"].dt.quarter
dim_date_df["year_number"] = dim_date_df["full_date"].dt.year

dim_date_df = dim_date_df.sort_values("full_date").reset_index(drop=True)
dim_date_df["full_date"] = dim_date_df["full_date"].dt.date

print("Dimenzija datuma:")
print(dim_date_df.shape)
display(dim_date_df.head())

Dimenzija datuma:
(31, 6)


,full_date,day_of_month,month_number,month_name,quarter_number,year_number
0,2025-01-01,1,1,January,1,2025
1,2025-01-02,2,1,January,1,2025
2,2025-01-03,3,1,January,1,2025
3,2025-01-04,4,1,January,1,2025
4,2025-01-05,5,1,January,1,2025


### Dimenzija kupca

In [89]:
dim_customer_df = etl_df[
    [
        "customer_id",
        "customer_code",
        "customer_name",
        "gender",
        "age",
        "customer_contact",
        "customer_email"
    ]
].drop_duplicates(subset=["customer_id"]).copy()

dim_customer_df["age_group"] = dim_customer_df["age"].apply(assign_age_group)

dim_customer_df["version"] = 1
dim_customer_df["date_from"] = date_from_value
dim_customer_df["date_to"] = None
dim_customer_df["is_current"] = 1

dim_customer_df = dim_customer_df[
    [
        "customer_id",
        "customer_code",
        "customer_name",
        "gender",
        "age",
        "age_group",
        "customer_contact",
        "customer_email",
        "version",
        "date_from",
        "date_to",
        "is_current"
    ]
]

print("Dimenzija kupca:")
print(dim_customer_df.shape)
display(dim_customer_df.head())

Dimenzija kupca:
(200000, 12)


,customer_id,customer_code,customer_name,gender,age,age_group,customer_contact,customer_email,version,date_from,date_to,is_current
0,1,d5f6ec07-d69e-4f47-b9b4-7c58ff17c19e,Osha Tella,Male,60,46-60,+9198579XXXXXX,oshaXXXXX@XXXXX.com,1,2025-01-01,None,1
1,2,3a73a0e5-d4da-45aa-85f3-528413900a35,Ekani Nazareth,Male,20,18-25,+9197745XXXXXX,ekaniXXX@XXXXXX.com,1,2025-01-01,None,1
2,3,6c870d65-76b0-431d-bdf3-9292998e8211,Ishanvi Dar,Male,54,46-60,+9198318XXXXXX,ishanviXXX@XXXXX.com,1,2025-01-01,None,1
3,4,5323737c-bbd2-423f-9c9b-e0433c8f75dc,Arya Shroff,Female,61,60+,+9194785XXXXXX,aryaXXX@XXXXX.com,1,2025-01-01,None,1
4,5,c0c3d474-f6c2-4c66-9b0e-f9ba75c6f310,Jackson Shere,Male,32,26-35,+9193423XXXXXX,jacksonXXX@XXXXXXX.com,1,2025-01-01,None,1


### Dimenzija računa

In [92]:
dim_account_df = etl_df[
    [
        "account_id",
        "account_type_name",
        "bank_branch_name",
        "city_name",
        "state_name"
    ]
].drop_duplicates(subset=["account_id"]).copy()

dim_account_df["version"] = 1
dim_account_df["date_from"] = date_from_value
dim_account_df["date_to"] = None
dim_account_df["is_current"] = 1

dim_account_df = dim_account_df[
    [
        "account_id",
        "account_type_name",
        "bank_branch_name",
        "city_name",
        "state_name",
        "version",
        "date_from",
        "date_to",
        "is_current"
    ]
]

print("Dimenzija računa:")
print(dim_account_df.shape)
display(dim_account_df.head())

Dimenzija računa:
(200000, 9)


,account_id,account_type_name,bank_branch_name,city_name,state_name,version,date_from,date_to,is_current
0,1,Savings,Thiruvananthapuram Branch,Thiruvananthapuram,Kerala,1,2025-01-01,None,1
1,2,Savings,Bhagalpur Branch,Bhagalpur,Bihar,1,2025-01-01,None,1
2,3,Checking,Ahmedabad Branch,Ahmedabad,Gujarat,1,2025-01-01,None,1
3,4,Business,New Delhi Branch,New Delhi,Delhi,1,2025-01-01,None,1
4,5,Business,Port Blair Branch,Port Blair,Andaman and Nicobar Islands,1,2025-01-01,None,1


### Dimenzija trgovca

In [95]:
dim_merchant_df = etl_df[
    [
        "merchant_id",
        "merchant_code",
        "merchant_category_name"
    ]
].drop_duplicates(subset=["merchant_id"]).copy()

dim_merchant_df["version"] = 1
dim_merchant_df["date_from"] = date_from_value
dim_merchant_df["date_to"] = None
dim_merchant_df["is_current"] = 1

dim_merchant_df = dim_merchant_df[
    [
        "merchant_id",
        "merchant_code",
        "merchant_category_name",
        "version",
        "date_from",
        "date_to",
        "is_current"
    ]
]

print("Dimenzija trgovca:")
print(dim_merchant_df.shape)
display(dim_merchant_df.head())

Dimenzija trgovca:
(200000, 7)


,merchant_id,merchant_code,merchant_category_name,version,date_from,date_to,is_current
0,1,214e03c5-5c34-40d1-a66c-f440aa2bbd02,Restaurant,1,2025-01-01,None,1
1,2,97977d83-5486-4510-af1c-8dada3e1cfa0,Groceries,1,2025-01-01,None,1
2,3,d82a798d-7b4d-4609-a687-8f6b5fc58fe7,Entertainment,1,2025-01-01,None,1
3,4,e0b802ba-b1d9-4e94-9a88-6fbb6066b4e0,Health,1,2025-01-01,None,1
4,5,ec344461-1bab-4e85-a1cb-05061e9149bc,Clothing,1,2025-01-01,None,1


### Dimenzija uređaja

In [98]:
dim_device_df = etl_df[
    [
        "transaction_device_id",
        "transaction_device_name",
        "device_type_name"
    ]
].drop_duplicates(subset=["transaction_device_id"]).copy()

print("Dimenzija uređaja:")
print(dim_device_df.shape)
display(dim_device_df.head())

Dimenzija uređaja:
(20, 3)


,transaction_device_id,transaction_device_name,device_type_name
0,1,Voice Assistant,POS
1,2,ATM,Desktop
2,3,Mobile Device,POS
3,4,Payment Gateway Device,Desktop
4,5,Debit/Credit Card,ATM


### Dimenzija vrste transakcije

In [101]:
dim_transaction_type_df = etl_df[
    [
        "transaction_type_id",
        "transaction_type_name"
    ]
].drop_duplicates(subset=["transaction_type_id"]).copy()

print("Dimenzija vrste transakcije:")
print(dim_transaction_type_df.shape)
display(dim_transaction_type_df.head())

Dimenzija vrste transakcije:
(5, 2)


,transaction_type_id,transaction_type_name
0,1,Transfer
1,2,Bill Payment
4,3,Debit
5,4,Withdrawal
6,5,Credit


### Dimenzija lokacije transakcije

In [104]:
dim_transaction_location_df = etl_df[
    [
        "transaction_location_id",
        "transaction_location_name"
    ]
].drop_duplicates(subset=["transaction_location_id"]).copy()

print("Dimenzija lokacije transakcije:")
print(dim_transaction_location_df.shape)
display(dim_transaction_location_df.head())

Dimenzija lokacije transakcije:
(148, 2)


,transaction_location_id,transaction_location_name
0,1,"Thiruvananthapuram, Kerala"
1,2,"Bhagalpur, Bihar"
2,3,"Ahmedabad, Gujarat"
3,4,"New Delhi, Delhi"
4,5,"Port Blair, Andaman and Nicobar Islands"


### Dimenzija valute

In [107]:
dim_currency_df = etl_df[
    [
        "currency_id",
        "currency_code"
    ]
].drop_duplicates(subset=["currency_id"]).copy()

print("Dimenzija valute:")
print(dim_currency_df.shape)
display(dim_currency_df.head())

Dimenzija valute:
(1, 2)


,currency_id,currency_code
0,1,INR


### Provjera izrađenih dimenzijskih dataframeova

In [110]:
dimension_summary = pd.DataFrame({
    "dimension": [
        "dim_date",
        "dim_customer",
        "dim_account",
        "dim_merchant",
        "dim_device",
        "dim_transaction_type",
        "dim_transaction_location",
        "dim_currency"
    ],
    "row_count": [
        len(dim_date_df),
        len(dim_customer_df),
        len(dim_account_df),
        len(dim_merchant_df),
        len(dim_device_df),
        len(dim_transaction_type_df),
        len(dim_transaction_location_df),
        len(dim_currency_df)
    ]
})

display(dimension_summary)

,dimension,row_count
0,dim_date,31
1,dim_customer,200000
2,dim_account,200000
3,dim_merchant,200000
4,dim_device,20
5,dim_transaction_type,5
6,dim_transaction_location,148
7,dim_currency,1


In [112]:
duplicate_checks = {
    "dim_date": dim_date_df["full_date"].duplicated().sum(),
    "dim_customer": dim_customer_df["customer_id"].duplicated().sum(),
    "dim_account": dim_account_df["account_id"].duplicated().sum(),
    "dim_merchant": dim_merchant_df["merchant_id"].duplicated().sum(),
    "dim_device": dim_device_df["transaction_device_id"].duplicated().sum(),
    "dim_transaction_type": dim_transaction_type_df["transaction_type_id"].duplicated().sum(),
    "dim_transaction_location": dim_transaction_location_df["transaction_location_id"].duplicated().sum(),
    "dim_currency": dim_currency_df["currency_id"].duplicated().sum()
}

duplicate_checks_df = pd.DataFrame(
    duplicate_checks.items(),
    columns=["dimension", "duplicate_count"]
)

display(duplicate_checks_df)

,dimension,duplicate_count
0,dim_date,0
1,dim_customer,0
2,dim_account,0
3,dim_merchant,0
4,dim_device,0
5,dim_transaction_type,0
6,dim_transaction_location,0
7,dim_currency,0


## Load faza - punjenje dimenzijskih tablica

In [114]:
tables_to_truncate = [
    "fact_bank_transaction",
    "dim_customer",
    "dim_account",
    "dim_merchant",
    "dim_device",
    "dim_transaction_type",
    "dim_transaction_location",
    "dim_currency",
    "dim_date"
]

with dw_engine.begin() as connection:
    connection.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
    
    for table_name in tables_to_truncate:
        connection.execute(text(f"TRUNCATE TABLE {table_name};"))
    
    connection.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))

print("Dimenzijske tablice i tablica činjenica su ispražnjene prije novog punjenja.")

Dimenzijske tablice i tablica činjenica su ispražnjene prije novog punjenja.


### Punjenje dimenzijskih tablica

In [116]:
dim_date_df.to_sql(
    "dim_date",
    con=dw_engine,
    if_exists="append",
    index=False,
    chunksize=5000,
    method="multi"
)

dim_customer_df.to_sql(
    "dim_customer",
    con=dw_engine,
    if_exists="append",
    index=False,
    chunksize=5000,
    method="multi"
)

dim_account_df.to_sql(
    "dim_account",
    con=dw_engine,
    if_exists="append",
    index=False,
    chunksize=5000,
    method="multi"
)

dim_merchant_df.to_sql(
    "dim_merchant",
    con=dw_engine,
    if_exists="append",
    index=False,
    chunksize=5000,
    method="multi"
)

dim_device_df.to_sql(
    "dim_device",
    con=dw_engine,
    if_exists="append",
    index=False,
    chunksize=5000,
    method="multi"
)

dim_transaction_type_df.to_sql(
    "dim_transaction_type",
    con=dw_engine,
    if_exists="append",
    index=False,
    chunksize=5000,
    method="multi"
)

dim_transaction_location_df.to_sql(
    "dim_transaction_location",
    con=dw_engine,
    if_exists="append",
    index=False,
    chunksize=5000,
    method="multi"
)

dim_currency_df.to_sql(
    "dim_currency",
    con=dw_engine,
    if_exists="append",
    index=False,
    chunksize=5000,
    method="multi"
)

print("Dimenzijske tablice su uspješno napunjene.")

Dimenzijske tablice su uspješno napunjene.


### Provjera punjenja dimenzijskih tablica

In [118]:
expected_dimension_counts = {
    "dim_date": len(dim_date_df),
    "dim_customer": len(dim_customer_df),
    "dim_account": len(dim_account_df),
    "dim_merchant": len(dim_merchant_df),
    "dim_device": len(dim_device_df),
    "dim_transaction_type": len(dim_transaction_type_df),
    "dim_transaction_location": len(dim_transaction_location_df),
    "dim_currency": len(dim_currency_df)
}

load_check_results = []

with dw_engine.connect() as connection:
    for table_name, expected_count in expected_dimension_counts.items():
        query = text(f"SELECT COUNT(*) AS row_count FROM {table_name};")
        actual_count = pd.read_sql(query, connection)["row_count"].iloc[0]
        
        load_check_results.append({
            "table_name": table_name,
            "expected_count": expected_count,
            "actual_count": actual_count,
            "difference": actual_count - expected_count
        })

load_check_df = pd.DataFrame(load_check_results)

display(load_check_df)

,table_name,expected_count,actual_count,difference
0,dim_date,31,31,0
1,dim_customer,200000,200000,0
2,dim_account,200000,200000,0
3,dim_merchant,200000,200000,0
4,dim_device,20,20,0
5,dim_transaction_type,5,5,0
6,dim_transaction_location,148,148,0
7,dim_currency,1,1,0


In [120]:
if (load_check_df["difference"] == 0).all():
    print("Sve dimenzijske tablice uspješno su napunjene očekivanim brojem zapisa.")
else:
    print("Postoji razlika u broju zapisa. Potrebno je provjeriti punjenje dimenzija.")

Sve dimenzijske tablice uspješno su napunjene očekivanim brojem zapisa.


## Load faza - priprema tablice činjenica

In [123]:
with dw_engine.connect() as connection:
    dim_date_map = pd.read_sql(
        "SELECT date_tk, full_date FROM dim_date;",
        connection
    )
    
    dim_customer_map = pd.read_sql(
        "SELECT customer_tk, customer_id FROM dim_customer WHERE is_current = 1;",
        connection
    )
    
    dim_account_map = pd.read_sql(
        "SELECT account_tk, account_id FROM dim_account WHERE is_current = 1;",
        connection
    )
    
    dim_merchant_map = pd.read_sql(
        "SELECT merchant_tk, merchant_id FROM dim_merchant WHERE is_current = 1;",
        connection
    )
    
    dim_device_map = pd.read_sql(
        "SELECT device_tk, transaction_device_id FROM dim_device;",
        connection
    )
    
    dim_transaction_type_map = pd.read_sql(
        "SELECT transaction_type_tk, transaction_type_id FROM dim_transaction_type;",
        connection
    )
    
    dim_transaction_location_map = pd.read_sql(
        "SELECT transaction_location_tk, transaction_location_id FROM dim_transaction_location;",
        connection
    )
    
    dim_currency_map = pd.read_sql(
        "SELECT currency_tk, currency_id FROM dim_currency;",
        connection
    )

dim_date_map["full_date"] = pd.to_datetime(dim_date_map["full_date"]).dt.date

print("Surogat ključevi iz dimenzija su dohvaćeni.")

Surogat ključevi iz dimenzija su dohvaćeni.


### Povezivanje transakcija s dimenzijama

In [126]:
fact_prepared_df = etl_df[
    [
        "transaction_id",
        "transaction_code",
        "transaction_date",
        "customer_id",
        "account_id",
        "merchant_id",
        "transaction_device_id",
        "transaction_type_id",
        "transaction_location_id",
        "currency_id",
        "transaction_time",
        "transaction_amount",
        "account_balance",
        "is_fraud",
        "transaction_description"
    ]
].copy()

fact_prepared_df["full_date"] = pd.to_datetime(
    fact_prepared_df["transaction_date"]
).dt.date

fact_prepared_df = fact_prepared_df.merge(
    dim_date_map,
    on="full_date",
    how="left",
    validate="many_to_one"
)

fact_prepared_df = fact_prepared_df.merge(
    dim_customer_map,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

fact_prepared_df = fact_prepared_df.merge(
    dim_account_map,
    on="account_id",
    how="left",
    validate="many_to_one"
)

fact_prepared_df = fact_prepared_df.merge(
    dim_merchant_map,
    on="merchant_id",
    how="left",
    validate="many_to_one"
)

fact_prepared_df = fact_prepared_df.merge(
    dim_device_map,
    on="transaction_device_id",
    how="left",
    validate="many_to_one"
)

fact_prepared_df = fact_prepared_df.merge(
    dim_transaction_type_map,
    on="transaction_type_id",
    how="left",
    validate="many_to_one"
)

fact_prepared_df = fact_prepared_df.merge(
    dim_transaction_location_map,
    on="transaction_location_id",
    how="left",
    validate="many_to_one"
)

fact_prepared_df = fact_prepared_df.merge(
    dim_currency_map,
    on="currency_id",
    how="left",
    validate="many_to_one"
)

print("Broj redaka i stupaca nakon povezivanja s dimenzijama:")
print(fact_prepared_df.shape)

display(fact_prepared_df.head())

Broj redaka i stupaca nakon povezivanja s dimenzijama:
(200000, 24)


,transaction_id,transaction_code,transaction_date,customer_id,account_id,merchant_id,transaction_device_id,transaction_type_id,transaction_location_id,currency_id,...,transaction_description,full_date,date_tk,customer_tk,account_tk,merchant_tk,device_tk,transaction_type_tk,transaction_location_tk,currency_tk
0,1,4fa3208f-9e23-42dc-b330-844829d0c12c,2025-01-23,1,1,1,1,1,1,1,...,Bitcoin transaction,2025-01-23,23,1,1,1,1,1,1,1
1,2,e41c55f9-c016-4ff3-872b-cae72467c75c,2025-01-25,2,2,2,2,2,2,1,...,Mutual fund investment,2025-01-25,25,2,2,2,2,2,2,1
2,3,af5f667c-d064-4083-bfb7-83396111a3da,2025-01-25,3,3,3,3,1,3,1,...,Seminar registration,2025-01-25,25,3,3,3,3,1,3,1
3,4,b1355810-d246-4aeb-9932-347f32646172,2025-01-04,4,4,4,4,1,4,1,...,Public transport pass,2025-01-04,4,4,4,4,4,1,4,1
4,5,c86a000c-d81f-40be-acdf-77fc072fd808,2025-01-16,5,5,5,5,3,5,1,...,Online shopping,2025-01-16,16,5,5,5,5,3,5,1


### Provjera stranih ključeva

In [128]:
foreign_key_columns = [
    "date_tk",
    "customer_tk",
    "account_tk",
    "merchant_tk",
    "device_tk",
    "transaction_type_tk",
    "transaction_location_tk",
    "currency_tk"
]

missing_foreign_keys = fact_prepared_df[foreign_key_columns].isna().sum()

print("Nedostajući surogat ključevi:")
display(missing_foreign_keys)

Nedostajući surogat ključevi:


date_tk                    0
customer_tk                0
account_tk                 0
merchant_tk                0
device_tk                  0
transaction_type_tk        0
transaction_location_tk    0
currency_tk                0
dtype: int64

### Izrada dataframea za tablicu činjenica

In [130]:
fact_prepared_df["transaction_count"] = 1

fact_bank_transaction_df = fact_prepared_df[
    [
        "transaction_id",
        "transaction_code",
        "date_tk",
        "customer_tk",
        "account_tk",
        "merchant_tk",
        "device_tk",
        "transaction_type_tk",
        "transaction_location_tk",
        "currency_tk",
        "transaction_time",
        "transaction_amount",
        "account_balance",
        "is_fraud",
        "transaction_count",
        "transaction_description"
    ]
].copy()

print("Konačan dataframe za tablicu činjenica:")
print(fact_bank_transaction_df.shape)

display(fact_bank_transaction_df.head())

Konačan dataframe za tablicu činjenica:
(200000, 16)


,transaction_id,transaction_code,date_tk,customer_tk,account_tk,merchant_tk,device_tk,transaction_type_tk,transaction_location_tk,currency_tk,transaction_time,transaction_amount,account_balance,is_fraud,transaction_count,transaction_description
0,1,4fa3208f-9e23-42dc-b330-844829d0c12c,23,1,1,1,1,1,1,1,16:04:07,32415.45,74557.27,0,1,Bitcoin transaction
1,2,e41c55f9-c016-4ff3-872b-cae72467c75c,25,2,2,2,2,2,2,1,03:09:52,63062.56,66817.99,0,1,Mutual fund investment
2,3,af5f667c-d064-4083-bfb7-83396111a3da,25,3,3,3,3,1,3,1,06:49:53,9711.15,61258.85,0,1,Seminar registration
3,4,b1355810-d246-4aeb-9932-347f32646172,4,4,4,4,4,1,4,1,00:53:33,94677.01,36313.61,0,1,Public transport pass
4,5,c86a000c-d81f-40be-acdf-77fc072fd808,16,5,5,5,5,3,5,1,04:04:48,67704.28,16948.73,0,1,Online shopping


### Punjenje tablice činjenica

In [132]:
fact_bank_transaction_df.to_sql(
    "fact_bank_transaction",
    con=dw_engine,
    if_exists="append",
    index=False,
    chunksize=1000,
    method="multi"
)

print("Tablica činjenica fact_bank_transaction uspješno je napunjena.")

Tablica činjenica fact_bank_transaction uspješno je napunjena.


In [134]:
with dw_engine.connect() as connection:
    fact_count = pd.read_sql(
        "SELECT COUNT(*) AS row_count FROM fact_bank_transaction;",
        connection
    )["row_count"].iloc[0]

print("Broj redaka u etl_df:", len(etl_df))
print("Broj redaka u fact_bank_transaction:", fact_count)

if len(etl_df) == fact_count:
    print("Tablica činjenica uspješno je napunjena očekivanim brojem zapisa.")
else:
    print("Postoji razlika u broju zapisa u tablici činjenica.")

Broj redaka u etl_df: 200000
Broj redaka u fact_bank_transaction: 200000
Tablica činjenica uspješno je napunjena očekivanim brojem zapisa.


## Završna provjera unosa podataka

Provjere:
- broj transakcija u izvoru odgovara broju transakcija u tablici činjenica,
- nisu izgubljeni podaci tijekom ETL procesa,
- nema nedostajućih stranih ključeva,
- glavne mjere u izvoru i skladištu jednake,
- tablica činjenica može se ispravno povezati sa svim dimenzijama.

In [136]:
source_validation = {
    "source_row_count": len(etl_df),
    "source_transaction_amount_sum": round(etl_df["transaction_amount"].sum(), 2),
    "source_account_balance_sum": round(etl_df["account_balance"].sum(), 2),
    "source_fraud_count": int(etl_df["is_fraud"].sum())
}

with dw_engine.connect() as connection:
    warehouse_validation = pd.read_sql(
        """
        SELECT
            COUNT(*) AS warehouse_row_count,
            COUNT(DISTINCT transaction_id) AS warehouse_distinct_transactions,
            ROUND(SUM(transaction_amount), 2) AS warehouse_transaction_amount_sum,
            ROUND(SUM(account_balance), 2) AS warehouse_account_balance_sum,
            SUM(is_fraud) AS warehouse_fraud_count,
            SUM(transaction_count) AS warehouse_transaction_count_sum
        FROM fact_bank_transaction;
        """,
        connection
    )

validation_results = pd.DataFrame({
    "check": [
        "Broj redaka",
        "Broj jedinstvenih transakcija",
        "Zbroj iznosa transakcija",
        "Zbroj stanja računa",
        "Broj fraud transakcija",
        "Zbroj transaction_count"
    ],
    "source_value": [
        source_validation["source_row_count"],
        source_validation["source_row_count"],
        source_validation["source_transaction_amount_sum"],
        source_validation["source_account_balance_sum"],
        source_validation["source_fraud_count"],
        source_validation["source_row_count"]
    ],
    "warehouse_value": [
        warehouse_validation["warehouse_row_count"].iloc[0],
        warehouse_validation["warehouse_distinct_transactions"].iloc[0],
        warehouse_validation["warehouse_transaction_amount_sum"].iloc[0],
        warehouse_validation["warehouse_account_balance_sum"].iloc[0],
        warehouse_validation["warehouse_fraud_count"].iloc[0],
        warehouse_validation["warehouse_transaction_count_sum"].iloc[0]
    ]
})

validation_results["difference"] = (
    validation_results["warehouse_value"].astype(float) -
    validation_results["source_value"].astype(float)
)

display(validation_results)

,check,source_value,warehouse_value,difference
0,Broj redaka,2.000000e+05,2.000000e+05,0.0
1,Broj jedinstvenih transakcija,2.000000e+05,2.000000e+05,0.0
2,Zbroj iznosa transakcija,9.907603e+09,9.907603e+09,0.0
3,Zbroj stanja računa,1.048760e+10,1.048760e+10,0.0
4,Broj fraud transakcija,1.008800e+04,1.008800e+04,0.0
5,Zbroj transaction_count,2.000000e+05,2.000000e+05,0.0


### Provjera nedostajućih stranih ključeva u tablici činjenica

In [138]:
with dw_engine.connect() as connection:
    null_fk_check = pd.read_sql(
        """
        SELECT
            SUM(CASE WHEN date_tk IS NULL THEN 1 ELSE 0 END) AS missing_date_tk,
            SUM(CASE WHEN customer_tk IS NULL THEN 1 ELSE 0 END) AS missing_customer_tk,
            SUM(CASE WHEN account_tk IS NULL THEN 1 ELSE 0 END) AS missing_account_tk,
            SUM(CASE WHEN merchant_tk IS NULL THEN 1 ELSE 0 END) AS missing_merchant_tk,
            SUM(CASE WHEN device_tk IS NULL THEN 1 ELSE 0 END) AS missing_device_tk,
            SUM(CASE WHEN transaction_type_tk IS NULL THEN 1 ELSE 0 END) AS missing_transaction_type_tk,
            SUM(CASE WHEN transaction_location_tk IS NULL THEN 1 ELSE 0 END) AS missing_transaction_location_tk,
            SUM(CASE WHEN currency_tk IS NULL THEN 1 ELSE 0 END) AS missing_currency_tk
        FROM fact_bank_transaction;
        """,
        connection
    )

display(null_fk_check)

,missing_date_tk,missing_customer_tk,missing_account_tk,missing_merchant_tk,missing_device_tk,missing_transaction_type_tk,missing_transaction_location_tk,missing_currency_tk
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Provjera povezivanja tablice činjenica s dimenzijama

Može li se svaka transakcija iz tablice činjenica povezati sa svim pripadajućim dimenzijama?

In [142]:
with dw_engine.connect() as connection:
    join_check = pd.read_sql(
        """
        SELECT COUNT(*) AS joined_row_count
        FROM fact_bank_transaction f
        JOIN dim_date d 
            ON f.date_tk = d.date_tk
        JOIN dim_customer c 
            ON f.customer_tk = c.customer_tk
        JOIN dim_account a 
            ON f.account_tk = a.account_tk
        JOIN dim_merchant m 
            ON f.merchant_tk = m.merchant_tk
        JOIN dim_device dev 
            ON f.device_tk = dev.device_tk
        JOIN dim_transaction_type tt 
            ON f.transaction_type_tk = tt.transaction_type_tk
        JOIN dim_transaction_location tl 
            ON f.transaction_location_tk = tl.transaction_location_tk
        JOIN dim_currency cur 
            ON f.currency_tk = cur.currency_tk;
        """,
        connection
    )

joined_row_count = join_check["joined_row_count"].iloc[0]

print("Broj redaka u fact tablici:", fact_count)
print("Broj redaka nakon JOIN-a sa svim dimenzijama:", joined_row_count)

if fact_count == joined_row_count:
    print("Sve transakcije iz tablice činjenica uspješno se povezuju s dimenzijama.")
else:
    print("Postoji problem u povezivanju fact tablice s dimenzijama.")

Broj redaka u fact tablici: 200000
Broj redaka nakon JOIN-a sa svim dimenzijama: 200000
Sve transakcije iz tablice činjenica uspješno se povezuju s dimenzijama.


In [144]:
measure_tolerance = 0.01

all_measure_checks_passed = (
    validation_results["difference"].astype(float).abs() <= measure_tolerance
).all()

all_fk_checks_passed = (null_fk_check.iloc[0] == 0).all()
join_check_passed = fact_count == joined_row_count

if all_measure_checks_passed and all_fk_checks_passed and join_check_passed:
    print("Završna provjera je uspješna. Podaci su ispravno učitani u skladište i nema izgubljenih podataka.")
else:
    print("Završna provjera nije u potpunosti uspješna. Potrebno je dodatno provjeriti rezultate.")

Završna provjera je uspješna. Podaci su ispravno učitani u skladište i nema izgubljenih podataka.
